In [1]:
import pandas as pd

df = pd.read_csv("../data_raw/ecommerce_customer_churn_dataset.csv")

In [2]:
# 数值列缺失填充（因为缺失代表行为未发生）
num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].fillna(0)

In [3]:
cat_cols = df.select_dtypes(exclude="number").columns
df[cat_cols] = df[cat_cols].fillna("Missing")

In [4]:
# Recency 风险特征
df["high_recency"] = (df["Days_Since_Last_Purchase"] >= 30).astype(int)

In [5]:
# 客服压力特征
df["service_pressure"] = (df["Customer_Service_Calls"] >= 5).astype(int)

In [6]:
# 低活跃特征
df["low_activity"] = (df["Login_Frequency"] <= 5).astype(int)

In [7]:
# 低参与特征
df["low_engagement"] = (
    (df["Pages_Per_Session"] <= 6) |
    (df["Session_Duration_Avg"] <= 20)
).astype(int)

In [8]:
# 价格敏感用户
df["price_sensitive"] = (
    (df["Discount_Usage_Rate"] >= 50) &
    (df["Average_Order_Value"] <= 120)
).astype(int)

In [9]:
# 综合风险指数
df["risk_score"] = (
    df["high_recency"] +
    df["service_pressure"] +
    df["low_activity"] +
    df["low_engagement"]
)


In [10]:
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter,high_recency,service_pressure,low_activity,low_engagement,price_sensitive,risk_score
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,953.33,2278.0,0,Q1,1,1,0,1,0,3
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,1067.47,3028.0,0,Q4,1,1,0,0,1,2
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,1289.75,2317.0,0,Q4,0,0,0,1,0,1
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,2340.92,2674.0,0,Q1,1,0,0,0,0,1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,0.0,19.1,9.0,...,3041.29,5354.0,0,Q4,1,0,0,1,0,2


In [11]:
df.describe()

,Age,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,Total_Purchases,Average_Order_Value,Days_Since_Last_Purchase,...,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,high_recency,service_pressure,low_activity,low_engagement,price_sensitive,risk_score
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,...,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.00000,50000.000000
mean,35.916600,2.984009,11.624660,25.780376,8.213542,57.079973,3.954520,13.111576,123.117330,28.005300,...,2.236180,1440.626292,1749.947600,0.289000,0.350220,0.641480,0.24382,0.433380,0.18142,1.668900
std,14.171222,2.059105,7.810657,12.594554,4.210062,16.282723,3.274206,7.017312,175.569714,29.647097,...,1.197375,907.249443,1309.277071,0.453302,0.477044,0.479571,0.42939,0.495547,0.38537,1.113327
min,0.000000,0.100000,0.000000,0.000000,0.000000,0.000000,0.000000,-13.000000,26.380000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0.000000
25%,28.000000,1.400000,6.000000,18.000000,5.500000,46.400000,1.000000,8.000000,87.050000,7.000000,...,1.000000,789.817500,647.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,1.000000
50%,37.000000,2.500000,11.000000,25.700000,8.100000,58.100000,3.000000,12.000000,112.970000,19.000000,...,2.000000,1243.415000,1697.000000,0.000000,0.000000,1.000000,0.00000,0.000000,0.00000,2.000000
75%,45.000000,4.000000,17.000000,34.000000,11.000000,68.700000,6.000000,17.000000,144.440000,39.000000,...,3.000000,1874.000000,2664.000000,1.000000,1.000000,1.000000,0.00000,1.000000,0.00000,2.000000
max,200.000000,10.000000,46.000000,75.600000,24.100000,143.743350,28.000000,128.700000,9666.379178,287.000000,...,5.000000,8987.240000,7197.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.00000,4.000000


In [12]:
df.groupby("risk_score")["Churned"].mean().sort_index()

risk_score
0    0.096831
1    0.203651
2    0.294027
3    0.466325
4    0.696549
Name: Churned, dtype: float64

In [13]:
df["risk_score"].value_counts(normalize=True).sort_index()

risk_score
0    0.15780
1    0.31338
2    0.28426
3    0.19124
4    0.05332
Name: proportion, dtype: float64

In [14]:
df["high_risk_user"] = (df["risk_score"] >= 3).astype(int)

df.groupby("high_risk_user")["Churned"].mean()

high_risk_user
0    0.215345
1    0.516519
Name: Churned, dtype: float64

In [15]:
import os
os.makedirs("data_clean", exist_ok=True)

# 保存清洗后的数据
df.to_csv("../data_clean/cleaned_data.csv", index=False)
print("✅ cleaned_data.csv 已生成！")

✅ cleaned_data.csv 已生成！
